In [6]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langgraph.store.memory import InMemoryStore
from langmem import create_manage_memory_tool, create_search_memory_tool
from dotenv import load_dotenv
import warnings
import os

warnings.filterwarnings("ignore")
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [7]:
manage_tool = create_manage_memory_tool(namespace=("user_123","memories"))
search_tool = create_search_memory_tool(namespace=("user_123" ,"memories"))

In [8]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.6,
    api_key=api_key
)
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [9]:
# Store:
store = InMemoryStore(
    index={
        "dims" : 384,
        "embed" : "huggingface:sentence-transformers/all-MiniLM-L6-v2",
        "fields" : ["content"]
    }
)
store

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1324.20it/s]


In [19]:
agent = create_agent(
    model=model,
    store=store,
    tools=[manage_tool,search_tool],
    system_prompt=
    """
   ou are a personal assistant with long-term memory.

Rules:
- When the user states a stable fact or preference, call manage_memory
  immediately, before replying.
- When the user asks anything about themselves ("tell me about me",
  "what do I prefer"), you MUST call search_memory first, then answer
  only from the results.
- Never claim you know nothing about the user without searching first.
- Do not store temporary statements or hypotheticals.
"""
)

In [20]:
config = {
    "configurable": {
        "thread_id": "memory-test-001"
    }
}

messages = [
    HumanMessage(content="I'm Arjun, 24 years old. I'm currently working as a junior data scientist at a startup in Bangalore."),

    HumanMessage(content="Mostly Python, SQL, and machine learning. I know TensorFlow, but lately I've been using PyTorch more. I initially preferred TensorFlow because its API felt easier, but now PyTorch is my first choice."),

    HumanMessage(content="I used to prefer video tutorials, but now I learn better by reading documentation and building projects. Short videos are still okay when the concept is difficult."),

    HumanMessage(content="One important thing: I don't like blindly following tutorials. I want to understand why an architecture or implementation decision is made."),

    HumanMessage(content="I was using a Windows laptop with 8GB RAM last year. I recently upgraded to 32GB RAM and an NVIDIA RTX 4060. I still use Windows."),

    HumanMessage(content="I'm building an AI-powered document assistant using FastAPI, PostgreSQL, and an LLM. Initially, I planned to use LangChain, but I'm now exploring LlamaIndex. I haven't completely abandoned LangChain."),

    HumanMessage(content="My current project deadline is December 15, 2026. That's just a temporary deadline, not a permanent preference for all my projects."),

    HumanMessage(content="I generally prefer PostgreSQL over MongoDB for structured application data. However, I wouldn't reject MongoDB if the use case justified it."),

    HumanMessage(content="My friend Ravi works with Java and Spring Boot. He prefers Java over Python. That's his preference, not mine."),

    HumanMessage(content="I don't like explanations that assume I'm already an expert. Even if I'm learning an advanced topic, explain unfamiliar concepts before using them."),

    HumanMessage(content="My long-term goal is to become an AI engineer who can design, build, deploy, and maintain complete AI systems. I don't want to restrict myself to model training alone."),

    HumanMessage(content="I used to think cloud deployment was unnecessary for my learning projects. I've changed my mind. I now consider deployment and monitoring important parts of learning AI engineering."),

    HumanMessage(content="I dislike unnecessary emojis and overly motivational language. Be direct, honest, and explain mistakes when I make them."),

    HumanMessage(content="Suppose I become a millionaire tomorrow, I would invest 40% in stocks, 30% in real estate, and 30% in a business. That's hypothetical, though; don't treat it as my actual financial portfolio."),

    HumanMessage(content="I am currently preparing for interviews, but I don't want this temporary interview preparation phase to define my permanent learning preferences."),

    HumanMessage(content="One more correction: I don't prefer lengthy explanations by default. However, when I ask 'deep dive', I expect mathematical intuition, implementation details, edge cases, and practical examples."),

    HumanMessage(content="I also enjoy cricket and kabaddi. I sometimes use sports analogies to understand technical concepts, but don't force them into every explanation."),

    HumanMessage(content="My preferred database for vector search is still undecided. I've experimented with pgvector and Chroma, but I haven't chosen a permanent favourite.")
]

In [12]:
for message in messages:

    response = agent.invoke({"messages" : [message]},config=config)
    print(response['messages'][-1].content)
    print("="*60)

Nice to meet you, Arjun! If there's anything specific you'd like me to remember or help you with—whether it’s data science projects, career advice, or anything else—just let me know.
Got it! Your background in Python, SQL, and machine learning, along with your recent switch to PyTorch, is now stored. If you need anything related to that, just let me know!
Got it! I’ve recorded that you prefer reading documentation and building projects for learning, but you still find short videos helpful when a concept is tricky. Let me know if there’s anything else you’d like me to remember or help with!
Got it! I’ll make sure to explain the reasoning behind any architectural or implementation choices we discuss. If you ever want a deeper dive into why a particular pattern or tool is chosen, just let me know and I’ll walk you through the trade‑offs and motivations.
Got it! Your Windows setup now has 32 GB of RAM and an NVIDIA RTX 4060. Let me know if you need any help with drivers, settings, or optim

In [13]:
def inspect_memories():
    memories = store.search(
        ("user_123", "memories"),
        limit=100
    )

    print("\n========== STORED MEMORIES ==========\n")

    for memory in memories:
        print("ID:", memory.key)
        print("CONTENT:", memory.value)
        print("SCORE:", memory.score)
        print("-" * 50)


inspect_memories()


========== STORED MEMORIES ==========

ID: 420ea36b-66ef-4832-8dbc-d92c4e01af3a
CONTENT: {'content': 'User: Arjun, 24, junior data scientist at a startup in Bangalore.'}
SCORE: None
--------------------------------------------------
ID: 2b7596bc-3420-4b52-b60c-14b634548620
CONTENT: {'content': 'User primarily works with Python, SQL, and machine learning. They have experience with TensorFlow but have recently shifted preference to PyTorch, citing its API as more favorable.'}
SCORE: None
--------------------------------------------------
ID: fa2fc79e-1954-4a9e-901c-a80429c4b14f
CONTENT: {'content': 'User prefers reading documentation and building projects over video tutorials, but finds short videos useful for difficult concepts.'}
SCORE: None
--------------------------------------------------
ID: 36c346a9-1c68-4fa7-8133-ae92e8d87aaf
CONTENT: {'content': 'User prefers to understand the reasoning behind architectural and implementation decisions rather than blindly following tutorials.'}

In [21]:
response = agent.invoke({"messages" : [{"role" : "user","content" : "Tell everything about me"}]}, config=config)

In [25]:
print(''.join(response['messages'][-1].content))

Here’s what I’ve stored about you:

- **Name & Age**: Arjun, 24 years old.  
- **Occupation**: Junior data scientist at a startup in Bangalore.  
- **Career goal**: Become an AI engineer capable of designing, building, deploying, and maintaining complete AI systems—not just training models.  
- **Learning style**:  
  - Prefers reading documentation and hands‑on projects over video tutorials.  
  - Short videos are acceptable for tricky concepts.  
  - Values explanations that don’t assume expert knowledge; wants unfamiliar concepts clarified first.  
  - Seeks concise answers by default, but will request a “deep dive” when needed (mathematics, implementation details, edge cases, practical examples).  
- **Technical interests**: Cloud deployment and monitoring are essential parts of your AI engineering learning path.  
- **Tool experience**: Have experimented with pgvector and Chroma for vector search but haven’t settled on a permanent favourite.  
- **Personal preferences**:  
  - Dis